# Step 1 — Load the Dataset

In [113]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/crop_prices_SA_raw.csv", parse_dates=["date"]) # load the data and parse the date column as date type
df 

,date,year,month,week_of_year,season,crop,region,price_per_kg,rainfall_mm,temp_max_c,temp_min_c,humidity_pct,supply_index
0,2020-01-01,2020,1,1,Summer,Maize,Free State,4.26,4.6,23.0,14.1,51.7,0.330
1,2020-01-08,2020,1,2,Summer,Maize,Free State,4.25,13.2,19.5,11.4,52.7,0.406
2,2020-01-15,2020,1,3,Summer,Maize,Free State,4.31,11.1,20.2,11.6,48.3,0.444
3,2020-01-22,2020,1,4,Summer,Maize,Free State,4.27,23.8,22.2,13.7,57.6,0.523
4,2020-01-29,2020,1,5,Summer,Maize,Free State,4.41,14.0,20.9,11.2,53.4,0.296
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5003,2025-11-26,2025,11,48,Spring,Wheat,North West,7.92,0.0,16.8,7.2,42.4,0.485
5004,2025-12-03,2025,12,49,Summer,Wheat,North West,7.72,0.0,19.7,11.0,27.6,0.551
5005,2025-12-10,2025,12,50,Summer,Wheat,North West,7.45,7.5,19.0,11.0,52.1,0.490
5006,2025-12-17,2025,12,51,Summer,Wheat,North West,7.40,3.9,18.2,10.0,38.8,0.507


# Step 2 : Display info and check for missing values.

In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5008 entries, 0 to 5007
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          5008 non-null   datetime64[ns]
 1   year          5008 non-null   int64         
 2   month         5008 non-null   int64         
 3   week_of_year  5008 non-null   int64         
 4   season        5008 non-null   object        
 5   crop          5008 non-null   object        
 6   region        5008 non-null   object        
 7   price_per_kg  4910 non-null   float64       
 8   rainfall_mm   4599 non-null   float64       
 9   temp_max_c    5008 non-null   float64       
 10  temp_min_c    4812 non-null   float64       
 11  humidity_pct  4762 non-null   float64       
 12  supply_index  4872 non-null   float64       
dtypes: datetime64[ns](1), float64(6), int64(3), object(3)
memory usage: 508.8+ KB


#### 2.1 check nulls

In [115]:
def display_info_and_missing_values(df):
    print("Null counts:\n", df.isnull().sum())
    print("\nNull percentage:\n", (df.isnull().sum() / len(df) * 100).round(2))


In [116]:
display_info_and_missing_values(df)

Null counts:
 date              0
year              0
month             0
week_of_year      0
season            0
crop              0
region            0
price_per_kg     98
rainfall_mm     409
temp_max_c        0
temp_min_c      196
humidity_pct    246
supply_index    136
dtype: int64

Null percentage:
 date            0.00
year            0.00
month           0.00
week_of_year    0.00
season          0.00
crop            0.00
region          0.00
price_per_kg    1.96
rainfall_mm     8.17
temp_max_c      0.00
temp_min_c      3.91
humidity_pct    4.91
supply_index    2.72
dtype: float64


#### 2.2 Handle nulls (for rainfall in millimetres, minimum temperature recorded in degrees Celsius, averege humidity percentage, market supply pressure for the crop and the price per kg)



##### For weather columns — fill with the median per region
##### (each region has its own climate so we don't mix them)

In [117]:
weather_cols = ["rainfall_mm", "temp_min_c", "humidity_pct"]
df[weather_cols] = df.groupby("region")[weather_cols].transform(
    lambda x: x.fillna(x.median())
)

display_info_and_missing_values(df[weather_cols])

Null counts:
 rainfall_mm     0
temp_min_c      0
humidity_pct    0
dtype: int64

Null percentage:
 rainfall_mm     0.0
temp_min_c      0.0
humidity_pct    0.0
dtype: float64


#####  For supply_index — fill with median per crop
##### (supply patterns differ per crop)

In [118]:

df["supply_index"] = df.groupby("crop")["supply_index"].transform(
    lambda x: x.fillna(x.median())
)

display_info_and_missing_values(df[["supply_index"]])

Null counts:
 supply_index    0
dtype: int64

Null percentage:
 supply_index    0.0
dtype: float64


#####  For price_per_kg — forward fill within each crop+region group
##### (use the last known price, most realistic for market data)

In [119]:
df["price_per_kg"] = df.groupby(["crop", "region"])["price_per_kg"].transform(
    lambda x: x.ffill().bfill()
)

display_info_and_missing_values(df[["price_per_kg"]])

Null counts:
 price_per_kg    0
dtype: int64

Null percentage:
 price_per_kg    0.0
dtype: float64


### Remaining nulls

In [120]:
print("Remaining nulls:\n", df.isnull().sum())

Remaining nulls:
 date            0
year            0
month           0
week_of_year    0
season          0
crop            0
region          0
price_per_kg    0
rainfall_mm     0
temp_max_c      0
temp_min_c      0
humidity_pct    0
supply_index    0
dtype: int64


# Step 3 — Feature Engineering
Im going to add 4 features:
1.	rolling_7d	- What was the average price this week — is today's price above or below the recent short-term norm?
2.	rolling_30d -	What was the average price over the last month — what is the medium-term trend?
3.	price_delta_7d - How fast is the price moving right now — is it accelerating up or down?
4.	price_delta_30d - Has the price been climbing or falling over the past month overall?

ref: https://www.geeksforgeeks.org/machine-learning/what-is-feature-engineering/

#### Sort first - critical for rolling calculations to be correct 
 - If the dates aren't in order, the rolling window picks up the wrong rows and the features become meaningless. Sorting by crop + region + date ensures each group's prices are in chronological order before the window slides across them.

In [121]:
df = df.sort_values(["crop", "region", "date"]).reset_index(drop=True)
df

,date,year,month,week_of_year,season,crop,region,price_per_kg,rainfall_mm,temp_max_c,temp_min_c,humidity_pct,supply_index
0,2020-01-01,2020,1,1,Summer,Maize,Free State,4.26,4.6,23.0,14.1,51.7,0.330
1,2020-01-08,2020,1,2,Summer,Maize,Free State,4.25,13.2,19.5,11.4,52.7,0.406
2,2020-01-15,2020,1,3,Summer,Maize,Free State,4.31,11.1,20.2,11.6,48.3,0.444
3,2020-01-22,2020,1,4,Summer,Maize,Free State,4.27,23.8,22.2,13.7,57.6,0.523
4,2020-01-29,2020,1,5,Summer,Maize,Free State,4.41,14.0,20.9,11.2,53.4,0.296
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5003,2025-11-26,2025,11,48,Spring,Wheat,North West,7.92,0.0,16.8,7.2,42.4,0.485
5004,2025-12-03,2025,12,49,Summer,Wheat,North West,7.72,0.0,19.7,11.0,27.6,0.551
5005,2025-12-10,2025,12,50,Summer,Wheat,North West,7.45,7.5,19.0,11.0,52.1,0.490
5006,2025-12-17,2025,12,51,Summer,Wheat,North West,7.40,3.9,18.2,10.0,38.8,0.507


###  Rolling averages (7 days and 30 days)
- We group by crop+region as Maize/Gauteng rolls separately from Wheat/FreeState

###   1. rolling_7d

In [122]:
df["rolling_7d"] = df.groupby(["crop", "region"])["price_per_kg"].transform(
    lambda x: x.rolling(window=2, min_periods=1).mean()
)


### 2. rolling_30d

In [123]:
df["rolling_30d"] = df.groupby(["crop", "region"])["price_per_kg"].transform(
    lambda x: x.rolling(window=4, min_periods=1).mean()
)


### 3. price_delta_7d (% change)

In [124]:
df["price_delta_7d"] = df.groupby(["crop", "region"])["price_per_kg"].transform(
    lambda x: x.pct_change(periods=1).mul(100).round(2)
)


### 4. price_delta_30d (% change)

In [125]:
df["price_delta_30d"] = df.groupby(["crop", "region"])["price_per_kg"].transform(
    lambda x: x.pct_change(periods=4).mul(100).round(2)
)

df.head(5)


,date,year,month,week_of_year,season,crop,region,price_per_kg,rainfall_mm,temp_max_c,temp_min_c,humidity_pct,supply_index,rolling_7d,rolling_30d,price_delta_7d,price_delta_30d
0,2020-01-01,2020,1,1,Summer,Maize,Free State,4.26,4.6,23.0,14.1,51.7,0.330,4.260,4.260000,NaN,NaN
1,2020-01-08,2020,1,2,Summer,Maize,Free State,4.25,13.2,19.5,11.4,52.7,0.406,4.255,4.255000,-0.23,NaN
2,2020-01-15,2020,1,3,Summer,Maize,Free State,4.31,11.1,20.2,11.6,48.3,0.444,4.280,4.273333,1.41,NaN
3,2020-01-22,2020,1,4,Summer,Maize,Free State,4.27,23.8,22.2,13.7,57.6,0.523,4.290,4.272500,-0.93,NaN
4,2020-01-29,2020,1,5,Summer,Maize,Free State,4.41,14.0,20.9,11.2,53.4,0.296,4.340,4.310000,3.28,3.52


In [126]:
df[["date", "crop", "region", "price_per_kg","rolling_7d", "rolling_30d","price_delta_7d", "price_delta_30d"]].head(10)

,date,crop,region,price_per_kg,rolling_7d,rolling_30d,price_delta_7d,price_delta_30d
0,2020-01-01,Maize,Free State,4.26,4.260,4.260000,NaN,NaN
1,2020-01-08,Maize,Free State,4.25,4.255,4.255000,-0.23,NaN
2,2020-01-15,Maize,Free State,4.31,4.280,4.273333,1.41,NaN
3,2020-01-22,Maize,Free State,4.27,4.290,4.272500,-0.93,NaN
4,2020-01-29,Maize,Free State,4.41,4.340,4.310000,3.28,3.52
5,2020-02-05,Maize,Free State,4.24,4.325,4.307500,-3.85,-0.24
6,2020-02-12,Maize,Free State,4.01,4.125,4.232500,-5.42,-6.96
7,2020-02-19,Maize,Free State,3.76,3.885,4.105000,-6.23,-11.94
8,2020-02-26,Maize,Free State,3.47,3.615,3.870000,-7.71,-21.32
9,2020-03-04,Maize,Free State,3.42,3.445,3.665000,-1.44,-19.34


In [127]:
print("\n----Null counts after feature engineering------:\n", df.isnull().sum())


----Null counts after feature engineering------:
 date                0
year                0
month               0
week_of_year        0
season              0
crop                0
region              0
price_per_kg        0
rainfall_mm         0
temp_max_c          0
temp_min_c          0
humidity_pct        0
supply_index        0
rolling_7d          0
rolling_30d         0
price_delta_7d     16
price_delta_30d    64
dtype: int64


### Remove the nulls from the new features added

In [128]:
# ── Drop rows where rolling/delta columns are null ─────────────
df = df.dropna(subset=["price_delta_7d", "price_delta_30d",
                        "rolling_7d", "rolling_30d"])

# ── Confirm ───────────────────────────────────────────────────
print("\nNulls after fix:\n", df.isnull().sum())
print("Remaining rows:", len(df))


Nulls after fix:
 date               0
year               0
month              0
week_of_year       0
season             0
crop               0
region             0
price_per_kg       0
rainfall_mm        0
temp_max_c         0
temp_min_c         0
humidity_pct       0
supply_index       0
rolling_7d         0
rolling_30d        0
price_delta_7d     0
price_delta_30d    0
dtype: int64
Remaining rows: 4944


## Step 4 — Generating the Direction Label(outcome)

In [129]:
# Generate direction label based on future price change (2 weeks ahead)
future_price = df.groupby(["crop", "region"])["price_per_kg"].transform(
    lambda x: x.shift(-2)
)

delta_forward = (future_price - df["price_per_kg"]) / df["price_per_kg"] * 100 # percentage change to determine direction

df["direction"] = pd.cut(
    delta_forward,
    bins=[-np.inf, -3, 3, np.inf], # thresholds for DOWN (<-3%), STABLE (-3% to 3%), UP (>3%)
    labels=["DOWN", "STABLE", "UP"]
)

# Drop last 2 rows of each group (no future price available)
df = df.dropna(subset=["direction"])

# Check label distribution
print("---------Direction counts--------------:\n", df["direction"].value_counts())
print("\n------Direction percentage-----------:\n",
      (df["direction"].value_counts() / len(df) * 100).round(2))

print("\n-------------------------\n")
display_info_and_missing_values(df[["direction"]])

---------Direction counts--------------:
 direction
STABLE    2545
UP        1237
DOWN      1130
Name: count, dtype: int64

------Direction percentage-----------:
 direction
STABLE    51.81
UP        25.18
DOWN      23.00
Name: count, dtype: float64

-------------------------

Null counts:
 direction    0
dtype: int64

Null percentage:
 direction    0.0
dtype: float64


C:\Users\sibus\AppData\Local\Temp\ipykernel_16124\2914229167.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["direction"] = pd.cut(


#### STATE OF THE DATAFRAME BEFORE ENCODING

In [130]:
df.head(10)

,date,year,month,week_of_year,season,crop,region,price_per_kg,rainfall_mm,temp_max_c,temp_min_c,humidity_pct,supply_index,rolling_7d,rolling_30d,price_delta_7d,price_delta_30d,direction
4,2020-01-29,2020,1,5,Summer,Maize,Free State,4.41,14.0,20.9,11.2,53.4,0.296,4.340,4.3100,3.28,3.52,DOWN
5,2020-02-05,2020,2,6,Summer,Maize,Free State,4.24,20.2,27.5,20.2,67.3,0.537,4.325,4.3075,-3.85,-0.24,DOWN
6,2020-02-12,2020,2,7,Summer,Maize,Free State,4.01,14.7,22.4,13.7,48.7,0.456,4.125,4.2325,-5.42,-6.96,DOWN
7,2020-02-19,2020,2,8,Summer,Maize,Free State,3.76,25.0,21.9,12.5,61.2,0.680,3.885,4.1050,-6.23,-11.94,DOWN
8,2020-02-26,2020,2,9,Summer,Maize,Free State,3.47,16.8,26.2,18.9,63.2,0.350,3.615,3.8700,-7.71,-21.32,STABLE
9,2020-03-04,2020,3,10,Autumn,Maize,Free State,3.42,21.2,33.3,26.0,49.4,0.525,3.445,3.6650,-1.44,-19.34,UP
10,2020-03-11,2020,3,11,Autumn,Maize,Free State,3.54,22.8,28.6,20.1,56.3,0.527,3.480,3.5475,3.51,-11.72,DOWN
11,2020-03-18,2020,3,12,Autumn,Maize,Free State,3.60,19.0,28.8,20.7,55.1,0.613,3.570,3.5075,1.69,-4.26,DOWN
12,2020-03-25,2020,3,13,Autumn,Maize,Free State,3.41,32.1,26.2,20.3,62.4,0.440,3.505,3.4925,-5.28,-1.73,DOWN
13,2020-04-01,2020,4,14,Autumn,Maize,Free State,3.20,44.2,34.1,26.5,76.4,0.522,3.305,3.4375,-6.16,-6.43,STABLE


## Step 5 — Encoding Categorical Columns

In [131]:
# Check which columns are categorical 
print("Column types:\n", df.dtypes)

Column types:
 date               datetime64[ns]
year                        int64
month                       int64
week_of_year                int64
season                     object
crop                       object
region                     object
price_per_kg              float64
rainfall_mm               float64
temp_max_c                float64
temp_min_c                float64
humidity_pct              float64
supply_index              float64
rolling_7d                float64
rolling_30d               float64
price_delta_7d            float64
price_delta_30d           float64
direction                category
dtype: object


In [132]:
#  Encode categorical columns 
le_crop = LabelEncoder()
le_region = LabelEncoder()
le_season = LabelEncoder()
le_direction = LabelEncoder()

df["crop"]      = le_crop.fit_transform(df["crop"])
df["region"]    = le_region.fit_transform(df["region"])
df["season"]    = le_season.fit_transform(df["season"])
df["direction"] = le_direction.fit_transform(df["direction"].astype(str))

#### STATE OF THE DATAFRAME AFTER CATEGORICAL COLUMNS ENCODING

In [137]:
df

,date,year,month,week_of_year,season,crop,region,price_per_kg,rainfall_mm,temp_max_c,temp_min_c,humidity_pct,supply_index,rolling_7d,rolling_30d,price_delta_7d,price_delta_30d,direction
4,2020-01-29,2020,1,5,2,0,0,4.41,14.0,20.9,11.2,53.4,0.296,4.340,4.3100,3.28,3.52,0
5,2020-02-05,2020,2,6,2,0,0,4.24,20.2,27.5,20.2,67.3,0.537,4.325,4.3075,-3.85,-0.24,0
6,2020-02-12,2020,2,7,2,0,0,4.01,14.7,22.4,13.7,48.7,0.456,4.125,4.2325,-5.42,-6.96,0
7,2020-02-19,2020,2,8,2,0,0,3.76,25.0,21.9,12.5,61.2,0.680,3.885,4.1050,-6.23,-11.94,0
8,2020-02-26,2020,2,9,2,0,0,3.47,16.8,26.2,18.9,63.2,0.350,3.615,3.8700,-7.71,-21.32,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5001,2025-11-12,2025,11,46,1,3,3,7.38,0.0,16.4,6.9,44.1,0.399,7.355,7.3225,0.68,4.68,2
5002,2025-11-19,2025,11,47,1,3,3,7.60,0.0,21.8,12.9,39.2,0.721,7.490,7.4100,2.98,4.83,1
5003,2025-11-26,2025,11,48,1,3,3,7.92,0.0,16.8,7.2,42.4,0.485,7.760,7.5575,4.21,8.05,0
5004,2025-12-03,2025,12,49,2,3,3,7.72,0.0,19.7,11.0,27.6,0.551,7.820,7.6550,-2.53,5.32,0


#### WHAT EACH COLUMN VALUE BECOMES

In [138]:
print("\nCrop encoding:     ", dict(zip(le_crop.classes_,
                                le_crop.transform(le_crop.classes_))))
print("Region encoding:   ", dict(zip(le_region.classes_,
                                le_region.transform(le_region.classes_))))
print("Season encoding:   ", dict(zip(le_season.classes_,
                                le_season.transform(le_season.classes_))))
print("Direction encoding:", dict(zip(le_direction.classes_,
                                le_direction.transform(le_direction.classes_))))


Crop encoding:      {'Maize': 0, 'Soybean': 1, 'Sunflower': 2, 'Wheat': 3}
Region encoding:    {'Free State': 0, 'Gauteng': 1, 'Mpumalanga': 2, 'North West': 3}
Season encoding:    {'Autumn': 0, 'Spring': 1, 'Summer': 2, 'Winter': 3}
Direction encoding: {'DOWN': 0, 'STABLE': 1, 'UP': 2}
